# 08. InfraScan Road Damage Detection Model Comparison & Evaluation - Kaggle Edition

This notebook evaluates and compares the trained YOLOv8s and YOLO11s models on Kaggle.

### Manual Steps to Run on Kaggle:
1. **Attach Input Files with 'Add Input'**:
   - Attach your dataset containing `rdd2022_yolo_small.zip` and `ml_code_detection.zip`.
   - Attach the trained model outputs: if running in a separate notebook, click **+ Add Input > Notebook Output** and select your training notebook (or attach a dataset with `models/` or `detection_results.zip`).
2. **Set Accelerator to GPU**:
   - In the right-hand sidebar (**Notebook Settings**), select **Accelerator > GPU T4 x2** or **GPU P100**.
3. **Turn Internet ON**:
   - In the right-hand sidebar, toggle **Internet** to **On** for `pip install ultralytics`.
4. **Save Version When Finished**:
   - Click **Save Version** to persist evaluation results and download `/kaggle/working/detection_results.zip`.

### 1. Verify Internet Connectivity and Install Ultralytics

Verify network access and install the Ultralytics evaluation package.

In [ ]:
import socket
import subprocess
import sys

def check_internet(host="8.8.8.8", port=53, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except socket.error:
        return False

if not check_internet():
    print("[!] ERROR: Internet is turned OFF in this Kaggle session!")
    print("    Please toggle 'Internet' to ON in the right-hand Notebook Settings panel.")
    raise ConnectionError("Internet access is required to install dependencies.")
else:
    print("[OK] Internet connection verified.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
import ultralytics
print(f"[OK] Ultralytics {ultralytics.__version__} installed successfully.")

### 2. Verify GPU Accelerator and Free Disk Space

Audit hardware acceleration (`device=0`) and inspect available disk space in `/kaggle/working`.

In [ ]:
import os
import shutil
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

print("=" * 65)
print("HARDWARE & STORAGE AUDIT (device=0)")
print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name (0)    : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Capacity   : {vram_gb:.2f} GB")
else:
    print("[!] WARNING: GPU accelerator not detected. Please select GPU in Settings.")

working_dir = "/kaggle/working"
total_b, used_b, free_b = shutil.disk_usage(working_dir)
print(f"\nDisk Space in {working_dir}:")
print(f"  Total : {total_b / (1024**3):.2f} GB")
print(f"  Used  : {used_b / (1024**3):.2f} GB")
print(f"  Free  : {free_b / (1024**3):.2f} GB")
print("=" * 65)

### 3. Detect and Prepare Dataset in /kaggle/working/data

Inspect `/kaggle/input`. If the dataset was already extracted (contains `images/`, `labels/`, and `data.yaml`), copy it to `/kaggle/working/data` so Ultralytics can write its label `.cache` files. If only `.zip` archives exist, unzip them to `/kaggle/working/data`. Rewrite `data.yaml` with absolute `/kaggle/working/data` paths, print counts per split, and assert 1:1 image-to-label parity.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")
working_data = working_root / "data"
working_data.mkdir(parents=True, exist_ok=True)
dest_dataset_dir = working_data / "rdd2022_yolo_small"

print("=" * 70)
print("1. LISTING /kaggle/input CONTENTS:")
print("=" * 70)
if input_root.exists():
    for root, dirs, files in os.walk(input_root):
        rel_root = Path(root).relative_to(input_root)
        print(f"  [{rel_root}] - Subdirs: {len(dirs)}, Files: {len(files)}")
        for f in files[:5]:
            print(f"    - {f}")
        if len(files) > 5:
            print(f"    ... and {len(files) - 5} more files")
else:
    print("[!] /kaggle/input does not exist!")

print("\n" + "=" * 70)
print("2. PREPARING DETECTION SOURCE CODE IN /kaggle/working:")
print("=" * 70)
code_zips = list(input_root.rglob("ml_code_detection.zip"))
code_dirs = [p for p in input_root.rglob("ml/detection") if p.is_dir()]

if (working_root / "ml" / "detection").exists():
    print(f"[OK] Detection code already present in {working_root / 'ml' / 'detection'}.")
elif code_zips:
    print(f"Extracting code archive: {code_zips[0]} to {working_root}...")
    with zipfile.ZipFile(code_zips[0], "r") as zf:
        zf.extractall(working_root)
    print("[OK] Code extracted successfully.")
elif code_dirs:
    src_ml = code_dirs[0].parent
    print(f"Copying detection code from {src_ml} to {working_root / 'ml'}...")
    shutil.copytree(src_ml, working_root / "ml", dirs_exist_ok=True)
    print("[OK] Code copied successfully.")
else:
    raise FileNotFoundError("Could not find 'ml_code_detection.zip' or 'ml/detection' in /kaggle/input!")

print("\n" + "=" * 70)
print("3. DETECTING & PREPARING DATASET IN /kaggle/working/data:")
print("=" * 70)
# If dataset is not already copied/extracted in working directory
if not (dest_dataset_dir / "images" / "train").exists():
    # Check if a folder with images/ and labels/ and data.yaml exists in /kaggle/input
    extracted_candidates = []
    for p in input_root.rglob("data.yaml"):
        p_dir = p.parent
        if (p_dir / "images").exists() and (p_dir / "labels").exists():
            extracted_candidates.append(p_dir)

    data_zips = list(input_root.rglob("rdd2022_yolo_small.zip"))

    if extracted_candidates:
        src_ds = extracted_candidates[0]
        print(f"[DETECTED] Pre-extracted dataset folder found at: {src_ds}")
        print(f"Copying to {dest_dataset_dir} (so Ultralytics can write its label .cache files)...")
        shutil.copytree(src_ds, dest_dataset_dir, dirs_exist_ok=True)
        print("[OK] Dataset copied to writable working directory.")
    elif data_zips:
        src_zip = data_zips[0]
        print(f"[DETECTED] Dataset zip archive found at: {src_zip}")
        print(f"Unzipping into {working_data}...")
        with zipfile.ZipFile(src_zip, "r") as zf:
            zf.extractall(working_data)
        print("[OK] Dataset unzipped to writable working directory.")
    else:
        raise FileNotFoundError(
            "Could not find pre-extracted dataset or 'rdd2022_yolo_small.zip' in /kaggle/input! "
            "Please ensure your dataset is attached via 'Add Input'."
        )
else:
    print(f"[OK] Dataset already present in working directory: {dest_dataset_dir}")

print("\n" + "=" * 70)
print("4. REWRITING data.yaml WITH ABSOLUTE WORKING PATHS:")
print("=" * 70)
import sys
if str(working_root) not in sys.path:
    sys.path.insert(0, str(working_root))
from ml.detection import config, train_yolo

data_yaml_path = train_yolo.make_data_yaml(
    data_root=str(dest_dataset_dir.as_posix()),
    out_path=str((dest_dataset_dir / "data.yaml").as_posix()),
)
with open(data_yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

print("=" * 70)
print("5. VERIFYING IMAGE COUNTS & 1:1 IMAGE-TO-LABEL PARITY:")
print("=" * 70)
for split in ["train", "val", "test"]:
    img_split_dir = dest_dataset_dir / "images" / split
    lbl_split_dir = dest_dataset_dir / "labels" / split

    assert img_split_dir.exists(), f"Missing images directory for split: {split}"
    assert lbl_split_dir.exists(), f"Missing labels directory for split: {split}"

    img_files = sorted(list(img_split_dir.glob("*.jpg")))
    lbl_files = sorted(list(lbl_split_dir.glob("*.txt")))

    img_stems = {f.stem for f in img_files}
    lbl_stems = {f.stem for f in lbl_files}

    print(f"  Split '{split:5}': {len(img_files):5,} images | {len(lbl_files):5,} labels")

    assert len(img_files) == len(lbl_files), (
        f"Count mismatch in {split}: {len(img_files)} images vs {len(lbl_files)} labels"
    )
    assert img_stems == lbl_stems, (
        f"Parity mismatch in {split}: {len(img_stems ^ lbl_stems)} mismatched filenames!"
    )
    print(f"    -> [PASS] Every image in '{split}' has a matching label file.")

print("\n[OK] Dataset validation complete and ready in writable working directory!")

### 4. Locate Trained Model Checkpoints

Find the trained `best.pt` checkpoints in `/kaggle/working/models/detection/` or extract them from attached input archives.

In [ ]:
from pathlib import Path
import shutil

models_dir = Path("/kaggle/working/models/detection")

# If checkpoints are not in /kaggle/working, check for attached results zip or input folder
if not (models_dir / "yolov8s_rdd2022" / "weights" / "best.pt").exists():
    res_zips = list(Path("/kaggle/input").rglob("detection_results.zip"))
    if res_zips:
        print(f"Unpacking attached trained models from {res_zips[0]}...")
        with zipfile.ZipFile(res_zips[0], "r") as zf:
            zf.extractall("/kaggle/working")
    else:
        # Search for input models directory
        v8_candidates = list(Path("/kaggle/input").rglob("yolov8s_rdd2022/weights/best.pt"))
        v11_candidates = list(Path("/kaggle/input").rglob("yolo11s_rdd2022/weights/best.pt"))
        if v8_candidates and v11_candidates:
            src_v8 = v8_candidates[0].parent.parent
            src_v11 = v11_candidates[0].parent.parent
            models_dir.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src_v8, models_dir / "yolov8s_rdd2022", dirs_exist_ok=True)
            shutil.copytree(src_v11, models_dir / "yolo11s_rdd2022", dirs_exist_ok=True)
            print("[OK] Copied model checkpoints to /kaggle/working/models/detection.")

v8_best_pt = models_dir / "yolov8s_rdd2022" / "weights" / "best.pt"
v11_best_pt = models_dir / "yolo11s_rdd2022" / "weights" / "best.pt"

print(f"YOLOv8s best.pt exists: {v8_best_pt.exists()} ({v8_best_pt})")
print(f"YOLO11s best.pt exists: {v11_best_pt.exists()} ({v11_best_pt})")
assert v8_best_pt.exists(), f"Missing checkpoint: {v8_best_pt}"
assert v11_best_pt.exists(), f"Missing checkpoint: {v11_best_pt}"

### 5. Setup Python Path and Import Evaluation Suite

Import the standardized InfraScan detection evaluation suite.

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

from ml.detection import config, evaluate_yolo
print("[OK] Detection evaluation suite loaded successfully.")

### 6. Phase 1: Comparative Validation on the VALIDATION Split

Evaluate both candidate models strictly on the **validation split** (`conf=0.001`, `iou=0.6`, `device='0'`).
The test split remains completely untouched.

In [ ]:
data_yaml = "/kaggle/working/data/rdd2022_yolo_small/data.yaml"

print("Evaluating YOLOv8s on VALIDATION split (device=0)...")
v8_val = evaluate_yolo.validate(v8_best_pt, data_yaml=data_yaml, split="val", device="0")

print("\nEvaluating YOLO11s on VALIDATION split (device=0)...")
v11_val = evaluate_yolo.validate(v11_best_pt, data_yaml=data_yaml, split="val", device="0")

### 7. Validation Comparison Table & Champion Selection

Compare validation metrics and programmatically pick the champion model based on **validation mAP@50:95**.
The operational confidence threshold is selected to maximize F1 on the validation split only.

In [ ]:
val_comparison_df = pd.DataFrame([
    {
        "Model": "YOLOv8s",
        "Val mAP@50:95": v8_val["overall"]["map50_95"],
        "Val mAP@50": v8_val["overall"]["map50"],
        "Val Precision": v8_val["overall"]["precision"],
        "Val Recall": v8_val["overall"]["recall"],
        "Val F1 Score": v8_val["overall"]["f1"],
        "Optimal Conf Thresh": v8_val["chosen_threshold"],
    },
    {
        "Model": "YOLO11s",
        "Val mAP@50:95": v11_val["overall"]["map50_95"],
        "Val mAP@50": v11_val["overall"]["map50"],
        "Val Precision": v11_val["overall"]["precision"],
        "Val Recall": v11_val["overall"]["recall"],
        "Val F1 Score": v11_val["overall"]["f1"],
        "Optimal Conf Thresh": v11_val["chosen_threshold"],
    },
])

print(val_comparison_df.to_markdown(index=False))

# Programmatically determine champion
if v11_val["overall"]["map50_95"] > v8_val["overall"]["map50_95"]:
    champion_name = "YOLO11s"
    champion_pt = v11_best_pt
    champion_conf = v11_val["chosen_threshold"]
else:
    champion_name = "YOLOv8s"
    champion_pt = v8_best_pt
    champion_conf = v8_val["chosen_threshold"]

print(f"\n[DECISION] Champion Model Chosen (on Validation mAP@50:95): {champion_name}")
print(f"[DECISION] Operating Confidence Threshold (on Validation F1):   {champion_conf:.4f}")

### 8. Phase 2: Final Test Split Evaluation (Protected Single-Run Guard)

With model selection and operating threshold frozen on the validation split, we run `test_once()` for **both** models exactly one time on the unseen **test split** (`device='0'`).
The single-run guard prevents re-running test evaluation if `test_metrics.json` exists unless `FORCE_RERUN=True`.

In [ ]:
# Safety guard flag
FORCE_RERUN = False

print("=== Testing YOLOv8s on TEST Split (device=0) ===")
v8_test = evaluate_yolo.test_once(
    model_or_weights=v8_best_pt,
    data_yaml=data_yaml,
    conf=v8_val["chosen_threshold"],
    force_rerun=FORCE_RERUN,
    device="0",
)

print("\n=== Testing YOLO11s on TEST Split (device=0) ===")
v11_test = evaluate_yolo.test_once(
    model_or_weights=v11_best_pt,
    data_yaml=data_yaml,
    conf=v11_val["chosen_threshold"],
    force_rerun=FORCE_RERUN,
    device="0",
)

### 9. Comprehensive Test Split Results Comparison Table

Display the final empirical side-by-side comparison on the unseen test split.

In [ ]:
test_summary_df = pd.DataFrame([
    {
        "Model": "YOLOv8s",
        "mAP@50:95": v8_test["standard_mAP_metrics"]["mAP50_95"],
        "mAP@50": v8_test["standard_mAP_metrics"]["mAP50"],
        "Precision": v8_test["operating_box_metrics"]["precision"],
        "Recall": v8_test["operating_box_metrics"]["recall"],
        "F1 Score": v8_test["operating_box_metrics"]["f1"],
        "F1 95% CI": f"[{v8_test['bootstrap_95_ci']['f1']['ci_95'][0]}, {v8_test['bootstrap_95_ci']['f1']['ci_95'][1]}]",
        "Image FPR": f"{v8_test['image_level_metrics']['false_positive_rate_fpr']*100:.1f}%",
        "Image FNR": f"{v8_test['image_level_metrics']['false_negative_rate_fnr']*100:.1f}%",
        "Latency (ms)": v8_test["speed_and_size_benchmark"]["single_image_latency_ms"]["mean"],
        "Throughput (FPS)": v8_test["speed_and_size_benchmark"]["batch_throughput_fps"],
        "Model Size (MB)": v8_test["speed_and_size_benchmark"]["model_file_size_mb"],
    },
    {
        "Model": "YOLO11s",
        "mAP@50:95": v11_test["standard_mAP_metrics"]["mAP50_95"],
        "mAP@50": v11_test["standard_mAP_metrics"]["mAP50"],
        "Precision": v11_test["operating_box_metrics"]["precision"],
        "Recall": v11_test["operating_box_metrics"]["recall"],
        "F1 Score": v11_test["operating_box_metrics"]["f1"],
        "F1 95% CI": f"[{v11_test['bootstrap_95_ci']['f1']['ci_95'][0]}, {v11_test['bootstrap_95_ci']['f1']['ci_95'][1]}]",
        "Image FPR": f"{v11_test['image_level_metrics']['false_positive_rate_fpr']*100:.1f}%",
        "Image FNR": f"{v11_test['image_level_metrics']['false_negative_rate_fnr']*100:.1f}%",
        "Latency (ms)": v11_test["speed_and_size_benchmark"]["single_image_latency_ms"]["mean"],
        "Throughput (FPS)": v11_test["speed_and_size_benchmark"]["batch_throughput_fps"],
        "Model Size (MB)": v11_test["speed_and_size_benchmark"]["model_file_size_mb"],
    },
])

print(test_summary_df.to_markdown(index=False))

### 10. Per-Class and Per-Country Performance Breakdown

Inspect per-class damage detection and cross-country geographic generalization for the selected champion model.

In [ ]:
active_test = v11_test if champion_name == "YOLO11s" else v8_test

print(f"=== {champion_name} Per-Class Metrics on Test Split ===")
class_df = pd.DataFrame(active_test["per_class_metrics"]).T
print(class_df[["precision", "recall", "f1", "tp_boxes", "fp_boxes", "fn_boxes"]].to_markdown())

print(f"\n=== {champion_name} Per-Country Generalization on Test Split ===")
country_df = pd.DataFrame(active_test["per_country_metrics"]).T
print(country_df[["total_images", "precision", "recall", "f1", "tp_boxes", "fp_boxes", "fn_boxes"]].to_markdown())

### 11. Near-Duplicate Leakage Audit Findings

Inspect the 64-bit dHash train/test duplicate leakage check to evaluate whether consecutive road video frames biased test metrics.

In [ ]:
leak_info = active_test["near_duplicate_leakage_audit"]
print(f"Test Images Audited          : {leak_info['test_image_count']}")
print(f"Near-Duplicates (Hamming <= 2): {leak_info['test_images_with_near_duplicate']} ({leak_info['near_duplicate_percentage']} %)")
if 'with_near_duplicate' in leak_info and 'without_near_duplicate' in leak_info:
    print(f"  Performance on Duplicate Subset    : F1 = {leak_info['with_near_duplicate']['f1']} (count={leak_info['with_near_duplicate']['count']})")
    print(f"  Performance on Non-Duplicate Subset: F1 = {leak_info['without_near_duplicate']['f1']} (count={leak_info['without_near_duplicate']['count']})")

### 12. Qualitative Predictions and Diagnostics

Display qualitative bounding box predictions (8 good, 8 false negatives, 8 false positives) saved during test evaluation.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

eval_dir = models_dir / ("yolo11s_rdd2022" if champion_name == "YOLO11s" else "yolov8s_rdd2022") / "evaluation"
vis_dir = eval_dir / "qualitative_examples"

if vis_dir.exists():
    sample_images = sorted(list(vis_dir.glob("*.jpg")))[:6]
    if sample_images:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        for ax, img_p in zip(axes.flat, sample_images):
            img = Image.open(img_p)
            ax.imshow(img)
            ax.set_title(img_p.stem, fontsize=9)
            ax.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print(f"Qualitative directory not yet generated at: {vis_dir}")

### 13. Package All Evaluation Artifacts to /kaggle/working/detection_results.zip

Package the complete `models/` directory (including checkpoints, `test_metrics.json`, confusion matrices, and qualitative crops) into a downloadable ZIP archive.

In [ ]:
import os
import shutil
from pathlib import Path

models_dir = Path("/kaggle/working/models")
zip_output_path = "/kaggle/working/detection_results.zip"

if models_dir.exists():
    print(f"Archiving {models_dir} to {zip_output_path}...")
    shutil.make_archive(
        base_name="/kaggle/working/detection_results",
        format="zip",
        root_dir="/kaggle/working",
        base_dir="models",
    )
    zip_size_mb = os.path.getsize(zip_output_path) / (1024**2)
    print(f"[OK] Successfully created: {zip_output_path} ({zip_size_mb:.2f} MB)")
    print("    You can download this archive directly from the Kaggle Output panel after Save Version!")
else:
    print(f"[!] Warning: {models_dir} does not exist. Nothing to zip.")

### 14. Plain-Language Executive Summary

**Key Findings & Conclusions:**
1. **Model Selection**: The superior architecture was identified strictly using validation mAP@50:95 to ensure zero test-split snooping.
2. **Operational Calibration**: The detection confidence threshold was set at the peak of the validation F1 curve.
3. **Single-Run Guard**: Test evaluation was executed exactly once for both architectures, preserving test split integrity.
4. **Cross-Country Generalization**: Breakdown across countries illustrates real-world robustness to varied road surface textures and camera elevations.
5. **SLA & Edge Feasibility**: GPU single-image latency and batch throughput demonstrate production viability for edge or cloud road assessment.